# 04 · Policy-Aware Question Answering

**Rules:** answer **only** from retrieved policy evidence, quote it **verbatim**, cite the **`policy_doc_id`**, and refuse instead of guessing when the policies do not cover the question.

In [1]:
# --- setup: make the project package importable and show the run mode ---
import sys, os, json, warnings
from pathlib import Path
# library deprecation notices print install paths; hide them here and in parallel worker processes
os.environ["PYTHONWARNINGS"] = "ignore"; warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import src                                   # sets macOS OpenMP compatibility flags
from src import config
from src.llm import mode_label
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
sns.set_theme(style="whitegrid", palette="Set2"); plt.rcParams["figure.dpi"] = 110
OUT = config.OUTPUTS
print("Mode:", mode_label(), "| outputs ->", OUT.relative_to(ROOT))

Mode: LIVE (gpt-4o-mini + text-embedding-3-small) | outputs -> outputs


In [2]:
from src import documents
from src.retrieval import policy_index
from src.policy_qa import answer_policy, validate, SYSTEM, REFUSAL
full, chunks = documents.build_policy_docs()
pd.DataFrame(full)

,doc_id,title,text
0,POL_RETURNS,Returns policy,Returns are allowed within 7 days of delivery for unused products with original packaging. Discounted prod...
1,POL_EXCHANGE,Exchange policy,"Size exchanges are allowed within 10 days of delivery, subject to stock availability. Exchange is not avai..."
2,POL_SHIPPING,Shipping policy,Standard delivery usually takes 3 to 5 business days. Express delivery usually takes 1 to 2 business days ...
3,POL_REFUND,Refund policy,Refunds are processed after the returned item passes quality inspection. Prepaid refunds are generally iss...
4,POL_WARRANTY,Warranty policy,Watches and selected accessories carry a 6-month manufacturer warranty against manufacturing defects only....


In [3]:
pd.DataFrame(chunks)[["chunk_id", "title", "text"]]

,chunk_id,title,text
0,POL_RETURNS#1,Returns policy,Returns are allowed within 7 days of delivery for unused products with original packaging.
1,POL_RETURNS#2,Returns policy,Discounted products are returnable only if the discount is 30 percent or less.
2,POL_EXCHANGE#1,Exchange policy,"Size exchanges are allowed within 10 days of delivery, subject to stock availability."
3,POL_EXCHANGE#2,Exchange policy,Exchange is not available for intimate wear or final sale products.
4,POL_SHIPPING#1,Shipping policy,Standard delivery usually takes 3 to 5 business days.
5,POL_SHIPPING#2,Shipping policy,Express delivery usually takes 1 to 2 business days in serviceable locations.
6,POL_REFUND#1,Refund policy,Refunds are processed after the returned item passes quality inspection.
7,POL_REFUND#2,Refund policy,Prepaid refunds are generally issued within 5 to 7 business days.
8,POL_WARRANTY#1,Warranty policy,Watches and selected accessories carry a 6-month manufacturer warranty against manufacturing defects only.
9,POL_WARRANTY#2,Warranty policy,Physical damage is not covered.


The 5 policies are short (2 sentences each). Each one is indexed as **sentence-level chunks**, so retrieval can point to the exact rule. The LLM then receives the **full text** of every policy that had a relevant chunk, because the second sentence often holds the condition, for example the "30 percent" rule in POL_RETURNS.

## 1. Retrieval inspection on labelled policy questions

In [4]:
labelled = [("Can I return shoes after 10 days?", "POL_RETURNS"), ("Is a 40% discounted item returnable?", "POL_RETURNS"),
            ("Can I exchange a shirt for a bigger size?", "POL_EXCHANGE"), ("Can I exchange innerwear?", "POL_EXCHANGE"),
            ("How long does standard delivery take?", "POL_SHIPPING"), ("Is express shipping available?", "POL_SHIPPING"),
            ("When will I get my money back?", "POL_REFUND"), ("Do refunds need a quality check?", "POL_REFUND"),
            ("Does my watch have a warranty?", "POL_WARRANTY"), ("Is a cracked watch glass covered?", "POL_WARRANTY")]
idx = policy_index(); rows = []
for q, gold in labelled:
    hits = idx.search(q, k=3)
    rows.append({"question": q, "gold": gold, "top1": hits[0]["doc_id"], "top1_score": hits[0]["score"],
                 "hit@1": hits[0]["doc_id"] == gold, "hit@3": gold in [h["doc_id"] for h in hits]})
pol_eval = pd.DataFrame(rows); pol_eval.to_csv(OUT / "04_policy_retrieval.csv", index=False)
print(f"policy retrieval hit@1 = {pol_eval['hit@1'].mean():.0%}, hit@3 = {pol_eval['hit@3'].mean():.0%}")
pol_eval

policy retrieval hit@1 = 100%, hit@3 = 100%


,question,gold,top1,top1_score,hit@1,hit@3
0,Can I return shoes after 10 days?,POL_RETURNS,POL_RETURNS,0.5452,True,True
1,Is a 40% discounted item returnable?,POL_RETURNS,POL_RETURNS,0.7423,True,True
2,Can I exchange a shirt for a bigger size?,POL_EXCHANGE,POL_EXCHANGE,0.4451,True,True
3,Can I exchange innerwear?,POL_EXCHANGE,POL_EXCHANGE,0.5279,True,True
4,How long does standard delivery take?,POL_SHIPPING,POL_SHIPPING,0.6560,True,True
5,Is express shipping available?,POL_SHIPPING,POL_SHIPPING,0.4733,True,True
6,When will I get my money back?,POL_REFUND,POL_REFUND,0.4685,True,True
7,Do refunds need a quality check?,POL_REFUND,POL_REFUND,0.6417,True,True
8,Does my watch have a warranty?,POL_WARRANTY,POL_WARRANTY,0.6112,True,True
9,Is a cracked watch glass covered?,POL_WARRANTY,POL_WARRANTY,0.3446,True,True


**Calibrating the refusal threshold.** A question is answered only if its best policy chunk scores at least `MIN_POLICY_SIM`. The threshold must sit **below the weakest correct match** among the labelled questions, but **above the strongest out-of-scope question**, so that the assistant refuses rather than guesses:

In [5]:
out_of_scope = ["Do you price-match other websites?", "Can I pay with cryptocurrency?", "What is your loyalty points expiry?",
                "Do you offer gift wrapping?", "What are your store opening hours?"]
oos = pd.DataFrame([{"question": q, "top1": idx.search(q, k=1)[0]["doc_id"], "top1_score": idx.search(q, k=1)[0]["score"]} for q in out_of_scope])
print(f"weakest correct match : {pol_eval.top1_score.min():.3f}  ({pol_eval.loc[pol_eval.top1_score.idxmin(), 'question']})")
print(f"strongest out-of-scope: {oos.top1_score.max():.3f}  ({oos.loc[oos.top1_score.idxmax(), 'question']})")
print(f"MIN_POLICY_SIM        : {config.MIN_POLICY_SIM}  -> {'separates them' if oos.top1_score.max() < config.MIN_POLICY_SIM <= pol_eval.top1_score.min() else 'DOES NOT separate them - recalibrate'}")
oos

weakest correct match : 0.345  (Is a cracked watch glass covered?)
strongest out-of-scope: 0.296  (What are your store opening hours?)
MIN_POLICY_SIM        : 0.32  -> separates them


,question,top1,top1_score
0,Do you price-match other websites?,POL_RETURNS,0.2781
1,Can I pay with cryptocurrency?,POL_REFUND,0.1935
2,What is your loyalty points expiry?,POL_RETURNS,0.2725
3,Do you offer gift wrapping?,POL_RETURNS,0.2128
4,What are your store opening hours?,POL_EXCHANGE,0.2964


## 2. Grounded answers with citations and quotes

Prompt:

In [6]:
print(SYSTEM)

You answer customer questions about store policies using ONLY the POLICY EVIDENCE provided.
Rules:
- Do not use outside knowledge or assume rules that are not written in the evidence.
- Quote the exact sentence(s) you rely on, word for word.
- Cite each policy you use by its doc_id (e.g. POL_RETURNS).
- If the evidence does not fully answer the question, say what the policy does say and that the rest is
  not specified; set "covered" to false. Recommend contacting customer support in that case.
- Keep the answer to 2-4 sentences, starting with a direct yes / no / it depends when possible.
Return JSON only: {"answer": "...", "citations": ["POL_..."], "quotes": ["exact sentence"], "covered": true}


In [7]:
def show(r):
    print("Q:", r["question"]); print("trace:"); [print("   ", t) for t in r["trace"]]
    print("ANSWER:", r["answer"]); print("citations:", r["citations"], "| quotes:", r["quotes"], "| covered:", r.get("covered"))
    print("=" * 110)
for q in ["Can I return a discounted product after 7 days?", "I bought a jacket at 40% off. Can I return it within 5 days if unused?",
          "Can I exchange a T-shirt for a different size after 8 days?", "My watch strap broke after 2 months - is it covered?",
          "When will I get my refund for a prepaid order?"]:
    show(answer_policy(q))

Q: Can I return a discounted product after 7 days?
trace:
    retrieved policy chunks: POL_RETURNS#1(0.68), POL_RETURNS#2(0.61), POL_REFUND#1(0.42)
    validation: citations retrieved, quotes verbatim
ANSWER: It depends on the discount percentage. Returns are allowed within 7 days for unused products, but discounted products can only be returned if the discount is 30 percent or less.
citations: ['POL_RETURNS'] | quotes: ['Returns are allowed within 7 days of delivery for unused products with original packaging.', 'Discounted products are returnable only if the discount is 30 percent or less.'] | covered: True
Q: I bought a jacket at 40% off. Can I return it within 5 days if unused?
trace:
    retrieved policy chunks: POL_RETURNS#1(0.54), POL_RETURNS#2(0.50), POL_EXCHANGE#1(0.42)
    validation: citations retrieved, quotes verbatim
ANSWER: No, you cannot return the jacket because it was discounted by 40%.
citations: ['POL_RETURNS'] | quotes: ['Discounted products are returnable only if 

Q: When will I get my refund for a prepaid order?
trace:
    retrieved policy chunks: POL_REFUND#2(0.66), POL_REFUND#1(0.45), POL_SHIPPING#1(0.35)
    validation: citations retrieved, quotes verbatim
ANSWER: You will generally receive your refund for a prepaid order within 5 to 7 business days after the returned item passes quality inspection.
citations: ['POL_REFUND'] | quotes: ['Prepaid refunds are generally issued within 5 to 7 business days.'] | covered: True


The policy answers are **conservative**:
- They start with yes, no or "it depends", followed by the exact policy sentence and its `doc_id`.
- When the evidence only partly covers the question, the answer says what the policy *does* state, marks `covered = false`, and suggests contacting support. For example, the warranty policy does not say whether a broken strap is a defect or damage.

## 3. Questions the policies do not cover → refusal, no citation

In [8]:
for q in ["Do you price-match other websites?", "Can I pay with cryptocurrency?", "What is your loyalty points expiry?"]:
    show(answer_policy(q))

Q: Do you price-match other websites?
trace:
    retrieved policy chunks: POL_RETURNS#2(0.28), POL_EXCHANGE#1(0.24), POL_REFUND#1(0.19)
    best similarity below 0.32 -> refuse instead of guessing
ANSWER: I could not find a store policy that covers this question, so I can't give a reliable answer. Please contact customer support.
citations: [] | quotes: [] | covered: None
Q: Can I pay with cryptocurrency?
trace:
    retrieved policy chunks: POL_REFUND#2(0.19), POL_EXCHANGE#2(0.14), POL_EXCHANGE#1(0.11)
    best similarity below 0.32 -> refuse instead of guessing
ANSWER: I could not find a store policy that covers this question, so I can't give a reliable answer. Please contact customer support.
citations: [] | quotes: [] | covered: None
Q: What is your loyalty points expiry?
trace:
    retrieved policy chunks: POL_RETURNS#1(0.27), POL_EXCHANGE#1(0.25), POL_REFUND#2(0.24)
    best similarity below 0.32 -> refuse instead of guessing
ANSWER: I could not find a store policy that covers thi

## 4. The citation validator (what stops a hallucinated rule)

In [9]:
ev = [{"doc_id": "POL_RETURNS", "full_text": documents.load_policies()["POL_RETURNS"] if hasattr(documents, "load_policies") else full[0]["text"]}]
bad = {"answer": "Yes, returns are accepted within 30 days.", "citations": ["POL_RETURNS"], "quotes": ["Returns are allowed within 30 days of delivery"]}
wrong_doc = {"answer": "...", "citations": ["POL_GIFTCARD"], "quotes": []}
good = {"answer": "...", "citations": ["POL_RETURNS"], "quotes": ["Returns are allowed within 7 days of delivery for unused products with original packaging."]}
for name, a in [("invented quote", bad), ("unretrieved doc id", wrong_doc), ("correct", good)]:
    print(f"{name:20} -> issues: {validate(a, ev) or 'none'}")

invented quote       -> issues: ["quote not verbatim in cited policy: 'Returns are allowed within 30 days of delivery'"]
unretrieved doc id   -> issues: ['citation POL_GIFTCARD was not retrieved', 'no quote']
correct              -> issues: none


Any LLM answer that fails validation is replaced by a purely **extractive** answer: the best-matching policy sentence, quoted with its doc_id. A made-up rule therefore never reaches the customer.